## 'Ford 엔진 데이터 예지보전 파이프라인' 전처리 모듈 
### [Cell 1] 라이브러리 설정, 데이터 로드 및 분할 (단계 1)
로우 데이터(Raw Data)를 메모리에 적재하고, 정상/비정상 클래스 비율을 유지하며 시스템 검증용 데이터셋으로 분리합니다.

In [7]:
import os
import itertools
import random
from time import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.collections import EllipseCollection
from scipy.io.arff import loadarff
from sklearn.preprocessing import StandardScaler, RobustScaler
import tensorflow as tf

from pathlib import Path

def get_project_root() -> Path:
    current_path = Path.cwd()
    # 현재 경로를 포함하여 모든 상위 디렉토리를 순차적으로 탐색
    for parent in [current_path, *current_path.parents]:
        # 루트 마커 파일(PHM_Ford_Vibration.md) 존재 여부 검증
        if (parent / "PHM_Ford_Vibration.md").exists():
            return parent
            
    # 마커 파일을 찾지 못했을 때의 실패 시나리오 방어 (명시적 에러 발생)
    raise FileNotFoundError("시스템 루트 마커(PHM_Ford_Vibration.md)를 찾을 수 없습니다. 실행 환경을 확인하세요.")

# 1. 시스템 루트 동적 탐색 및 주요 디렉토리 매핑
ROOT_DIR = get_project_root()
DATA_DIR = ROOT_DIR / "dataset"
SAVE_DIR = ROOT_DIR / "Result"

# 2. 결과물 저장 폴더 구조 보장
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 3. 경로 매핑 디버깅
print(f"동적 탐색된 시스템 루트: {ROOT_DIR}")
print(f"데이터 탐색 경로: {DATA_DIR}")
print(f"결과물 저장 경로: {SAVE_DIR}")

# --- 아래부터는 기존 데이터 파이프라인 진행 ---
# 문자열 결합이 아닌 Path 객체의 '/' 연산자를 사용하여 OS 독립성 확보
train_fn = DATA_DIR / "FordA_TRAIN.arff"
test_fn = DATA_DIR / "FordA_TEST.arff"

# (이후 read_ariff 함수 및 데이터 로드 코드 동일)

def read_ariff(path):
    raw_data, meta = loadarff(path)
    cols = [x for x in meta]
    data2d = np.zeros([raw_data.shape[0],len(cols)])
    
    for i, col in zip(range(len(cols)),cols):
        data2d[:,i]=raw_data[col]
    return data2d

train = read_ariff(train_fn)
test = read_ariff(test_fn)

x_train_temp = train[:,:-1]
y_train_temp = train[:, -1] 
x_test = test[:, :-1]
y_test = test[:, -1] 

normal_x=x_train_temp[y_train_temp==1] 
abnormal_x=x_train_temp[y_train_temp==-1] 
normal_y=y_train_temp[y_train_temp==1] 
abnormal_y=y_train_temp[y_train_temp==-1] 

ind_x_normal = int(normal_x.shape[0]*0.8) 
ind_y_normal = int(normal_y.shape[0]*0.8) 
ind_x_abnormal = int(abnormal_x.shape[0]*0.8) 
ind_y_abnormal = int(abnormal_y.shape[0]*0.8) 

x_train = np.concatenate((normal_x[:ind_x_normal], abnormal_x[:ind_x_abnormal]), axis=0)
x_valid = np.concatenate((normal_x[ind_x_normal:], abnormal_x[ind_x_abnormal:]), axis=0)
y_train = np.concatenate((normal_y[:ind_y_normal], abnormal_y[:ind_y_abnormal]), axis=0)
y_valid = np.concatenate((normal_y[ind_y_normal:], abnormal_y[ind_y_abnormal:]), axis=0)

동적 탐색된 시스템 루트: /Users/alohyomora/dev/github/Hyundai-Autoever-Mobility-SW---SmartFactory-Study/motor_vibration_anomaly_detection
데이터 탐색 경로: /Users/alohyomora/dev/github/Hyundai-Autoever-Mobility-SW---SmartFactory-Study/motor_vibration_anomaly_detection/dataset
결과물 저장 경로: /Users/alohyomora/dev/github/Hyundai-Autoever-Mobility-SW---SmartFactory-Study/motor_vibration_anomaly_detection/Result
